# diagnose_r2: score distributions, training normals vs test normals

Read-only: nothing is refit or tuned, and test data is only scored. The printed tables are what I need; the plots are a bonus.

Variants for the **same** normal test records:
- **V1 train-encoding**: training-partition vocabulary + `scaler.pkl` (what the models were fitted on)
- **V2a v1-eval cell 17**: test-fit `LabelEncoder`, **unscaled** (what produced R1/R2)
- **V2b v1-eval cell 23**: test-fit `LabelEncoder`, scaled (what produced R3)
- **V3 v1 runtime**: proto/state = -1 (what `detector.py` v1 did with the integer-code `feature_meta.json`)

In [ ]:
import sys, json, pickle, hashlib, platform
from pathlib import Path
import numpy as np, pandas as pd, sklearn

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    while True:
        if (cur / "Python").is_dir() and (cur / "RustSniffer").is_dir():
            return cur
        if cur.parent == cur:
            raise RuntimeError("Could not find NIDS root (folder containing Python/ and RustSniffer/).")
        cur = cur.parent

NB_DIR       = Path.cwd()                                   # Python/Notebooks/src
PROJECT_ROOT = find_project_root(NB_DIR)
SRC_DIR      = PROJECT_ROOT / "Python" / "src"
MODEL_DIR    = SRC_DIR / "models"                           # the ONLY artefact dir v2 uses (detector.py loads from here)
DATA_DIR     = NB_DIR / "datasets"
TRAIN_CSV    = DATA_DIR / "UNSW_NB15_training_set.csv"
TEST_CSV     = DATA_DIR / "UNSW_NB15_testing_set.csv"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from preprocessing import FEATURE_COLS, CAT_COLS

def sha12(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()[:12] if p.exists() else "MISSING"

print("python", platform.python_version(), "| sklearn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR   :", MODEL_DIR)
for f in ["scaler.pkl", "isolation_forest.pkl", "local_outlier_factor.pkl", "one_class_svm.pkl", "feature_meta.json"]:
    print(f"  {f:26s} {sha12(MODEL_DIR / f)}")
OLD_DIR = PROJECT_ROOT / "Python" / "Notebooks" / "Python" / "models"   # where v1 cell 8 / evaluations.ipynb also read and wrote
if OLD_DIR.exists():
    print("Second model dir exists (v1 leftover):", OLD_DIR)
    for f in ["scaler.pkl", "isolation_forest.pkl", "local_outlier_factor.pkl", "one_class_svm.pkl"]:
        print(f"  {f:26s} {sha12(OLD_DIR / f)}")

import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
with open(MODEL_DIR / "scaler.pkl", "rb") as f: scaler = pickle.load(f)
models = {}
for n, f in [("iforest", "isolation_forest.pkl"), ("lof", "local_outlier_factor.pkl"), ("ocsvm", "one_class_svm.pkl")]:
    with open(MODEL_DIR / f, "rb") as fh: models[n] = pickle.load(fh)
train = pd.read_csv(TRAIN_CSV); test = pd.read_csv(TEST_CSV)

train_maps = {c: {v: i for i, v in enumerate(sorted(train[c].astype(str).unique()))} for c in CAT_COLS}
def build(df, maps):
    X = df[FEATURE_COLS].copy()
    for c in CAT_COLS:
        X[c] = df[c].astype(str).map(maps[c]).fillna(-1.0)
    return X.apply(pd.to_numeric, errors="coerce").fillna(0).astype(float).to_numpy()
def v1_eval_encode(df_full, rows):
    X = df_full[FEATURE_COLS].copy()
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))   # fit on the WHOLE test frame, as v1 did
    return X.to_numpy(dtype=float)[rows]

tr_n = (train["label"] == 0).to_numpy(); te_n = (test["label"] == 0).to_numpy()
Z_tr = scaler.transform(build(train[tr_n], train_maps))
print("train normals", Z_tr.shape, "| matches LOF _fit_X exactly:", Z_tr.shape == models["lof"]._fit_X.shape and np.allclose(Z_tr, models["lof"]._fit_X, rtol=0, atol=1e-9))
for c in CAT_COLS:
    print(f"{c}: train {train[c].nunique()} labels, test {test[c].nunique()}; only-in-train {sorted(set(train[c].astype(str)) - set(test[c].astype(str)))}; only-in-test {sorted(set(test[c].astype(str)) - set(train[c].astype(str)))}")

: 

In [ ]:
X_v2a = v1_eval_encode(test, te_n)
variants = {
    "V1 train-encoding": scaler.transform(build(test[te_n], train_maps)),
    "V2a v1-eval c17 (unscaled)": X_v2a,
    "V2b v1-eval c23 (scaled)": scaler.transform(X_v2a),
    "V3 v1 runtime (-1)": scaler.transform(build(test[te_n], {c: {} for c in CAT_COLS})),
}
scores = {"train normals": {
    "iforest": models["iforest"].decision_function(Z_tr),
    "lof": models["lof"].negative_outlier_factor_ - models["lof"].offset_,   # unbiased training score for novelty LOF
    "ocsvm": models["ocsvm"].decision_function(Z_tr)}}
for v, Xv in variants.items():
    scores[v] = {n: m.decision_function(Xv) for n, m in models.items()}

rows = []
for s, d in scores.items():
    for n, sc in d.items():
        rows.append([s, n, len(sc), round(float((sc < 0).mean()), 4), *np.round(np.percentile(sc, [1, 25, 50, 75, 99]), 4)])
print("decision_function < 0  =>  flagged as anomaly")
print(pd.DataFrame(rows, columns=["set", "model", "n", "flag_rate", "p1", "p25", "p50", "p75", "p99"]).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, n in zip(axes, models):
    sets = ["train normals", "V1 train-encoding", "V2b v1-eval c23 (scaled)"]
    allv = np.concatenate([scores[s][n] for s in sets]); lo, hi = np.percentile(allv, [0.5, 99.5])
    bins = np.linspace(lo, hi, 80)
    ax.hist(np.clip(scores[sets[0]][n], lo, hi), bins=bins, density=True, alpha=.45, label=sets[0])
    ax.hist(np.clip(scores[sets[1]][n], lo, hi), bins=bins, density=True, histtype="step", lw=2, label=sets[1])
    ax.hist(np.clip(scores[sets[2]][n], lo, hi), bins=bins, density=True, histtype="step", lw=2, ls="--", label=sets[2])
    ax.axvline(0, color="k", lw=1); ax.set_title(f"{n}: decision_function (<0 = anomaly)")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()
print("V2a (unscaled) is left off the plot: its scores sit far outside the other ranges; see the table.")

## Drift of normal traffic between the official partitions (V1 encoding)
After scaling, the training normals have mean 0 and std 1 in every column. A large |mean| on the test normals is genuine distribution shift, separate from any pipeline bug.

In [ ]:
Zv = variants["V1 train-encoding"]
drift = pd.DataFrame({"feature": FEATURE_COLS, "test_mean_z": Zv.mean(0), "test_std_z": Zv.std(0), "frac_|z|>5": (np.abs(Zv) > 5).mean(0)})
drift["abs_mean"] = drift["test_mean_z"].abs()
print(drift.sort_values("abs_mean", ascending=False).head(12).round(3).to_string(index=False))